# Практика 2/3: Hugging Face — pipeline + load_dataset

> **Курс «От нуля до своих агентов» — Модуль 1.5.**
> Вторая из трёх практик. На этой задаче вы трогаете **только Hugging
> Face**: берёте готовую модель и готовый датасет с Hub'а, прогоняете
> модель по датасету, считаете accuracy.
>
> Время: ~10 минут.

## Что делаем

Hugging Face Hub — это **готовые куски** для NLP-задач. Маленький
specialized классификатор и стандартизованный датасет позволяют за 10
минут собрать рабочий пайплайн «классификация настроения», на который
в 2018-м уходила неделя.

Сделаем: возьмём `distilbert-base-uncased-finetuned-sst-2-english` из HF,
прогоним его по первым 50 отзывам из IMDB, сравним предсказания с
настоящими метками, посчитаем accuracy.

## Шаг 1. Установка и токен

Токен HF на этом упражнении не обязателен (модель и датасет публичные),
но если вы уже прикрепили `HF_TOKEN` в Secrets — отлично, ячейка ниже
его подцепит.

In [ ]:
!pip install -q transformers datasets

In [ ]:
# Если есть токен — используем; если нет, продолжаем без него.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF токен подключён.")
except Exception:
    HF_TOKEN = None
    print("Без HF токена — публичные модели всё равно доступны.")

## Шаг 2. Поднимаем модель — 3 строки

`pipeline("sentiment-analysis")` без аргументов скачивает дефолтную
модель (`distilbert-base-uncased-finetuned-sst-2-english`, ~250 МБ).
Первый запуск — минута, дальше — кэш.

In [ ]:
from transformers import pipeline

# device=-1 — CPU (distilbert крошечная, GPU не нужен).
# На Kaggle иногда дают Tesla P100, она несовместима со свежим
# PyTorch — поэтому явный CPU надёжнее.
clf = pipeline("sentiment-analysis", device=-1)

# Sanity check
print(clf("This course is unexpectedly fun."))
print(clf("Honestly, I'm just here for the certificate."))

## Шаг 3. Подключаем датасет — 2 строки

Берём `imdb` (50k отзывов о фильмах с разметкой POSITIVE/NEGATIVE),
из них первые 50 примеров теста — этого достаточно для прикидки.

In [ ]:
from datasets import load_dataset

ds = load_dataset("imdb", split="test[:50]")
print("Размер:", len(ds))
print("Поля:", ds.column_names)
print("\nПример:")
print(" текст:", ds[0]["text"][:200], "...")
print(" метка:", ds[0]["label"], "(0 = NEGATIVE, 1 = POSITIVE)")

## Шаг 4. Прогон модели по датасету + accuracy

**TODO:** ничего трогать не надо, просто запустите. После — переходите
к шагу 5.

In [ ]:
correct = 0
errors = []

for row in ds:
    text = row["text"][:512]  # обрезаем грубо по символам, чтобы влезло
    pred = clf(text)[0]["label"]
    true = "POSITIVE" if row["label"] == 1 else "NEGATIVE"
    if pred == true:
        correct += 1
    else:
        errors.append({"true": true, "pred": pred, "text": text[:120]})

acc = correct / len(ds)
print(f"Accuracy: {correct}/{len(ds)} = {acc:.0%}")
print(f"Ошибок: {len(errors)}")

## Шаг 5. Посмотрите на 3 ошибки глазами

**TODO:** напечатайте первые 3 случая, где модель ошиблась. Ваша
задача — глазами понять, **почему** модель могла сбиться.

In [ ]:
for i, e in enumerate(errors[:3], 1):
    print(f"--- Ошибка #{i} ---")
    print(f"Истина:    {e['true']}")
    print(f"Модель:    {e['pred']}")
    print(f"Текст:     {e['text']} ...")
    print()

## Шаг 6. Развитие — попробуйте другую модель

**TODO (опционально):** в одной строке кода поменяйте модель на
многоязычную и прогоните ещё раз.

In [ ]:
# TODO: раскомментируйте и сравните accuracy
# clf_multi = pipeline(
#     "sentiment-analysis",
#     model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
# )
# correct = sum(
#     1 for row in ds
#     if {"positive": "POSITIVE", "negative": "NEGATIVE", "neutral": "NEUTRAL"}.get(
#         clf_multi(row["text"][:512])[0]["label"], ""
#     ) == ("POSITIVE" if row["label"] == 1 else "NEGATIVE")
# )
# print(f"Multi-lingual accuracy: {correct/len(ds):.0%}")

## Шаг 7. Один абзац о наблюдении

**TODO:** одно-два предложения. На каких типах фраз distilbert сбоит?
Что общего у ошибок?

**TODO: ваше наблюдение здесь.**

## Сдача

`Save & Run All` → `Share Public` → ссылка в чат:
`[Модуль 1.5, практика HF] {ссылка}`.

Дальше — [практика 3: Google AI Studio](./practice-aistudio.ipynb).